## Trạng thái bản lưu trên GitHub

Xuất model của giai đoạn trước. Model web hiện tại được xuất bởi 06, không phải 05.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

# 05 — Final ME-STGCN-TA Deployment Model

Notebook này huấn luyện **một model duy nhất trên toàn bộ 160 video** để phục vụ demo video/web realtime.

Điểm cần phân biệt:

- Notebook 04b dùng Nested LOSO để đánh giá khả năng tổng quát hóa; mỗi checkpoint outer-fold chỉ học 3 subject.
- Notebook 05 dùng cả 4 subject để tạo model triển khai; vì không còn test độc lập, metric trên chính 160 video không được dùng làm kết quả đánh giá.
- Kết quả đánh giá chính vẫn là Nested LOSO 04b: sensitivity 0.9747, specificity 0.7901, FN=2, FP=17.

Cấu hình final được tổng hợp từ bốn outer folds: multiplier trung vị = 3.0, epoch trung vị = 12, threshold bảo thủ = 0.40.

In [ ]:
from pathlib import Path
import importlib.util
import json
import random
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.30

# Không tăng 30–50 epoch khi chưa có validation chứng minh.
# Bốn outer folds của 04b chọn: 17, 3, 13, 11 -> trung vị = 12.
FINAL_EPOCHS = 12
FALL_MULTIPLIER = 3.0
DECISION_THRESHOLD = 0.40

CHANNEL_NAMES = [
    "relative_x", "relative_y", "visibility",
    "joint_velocity_x", "joint_velocity_y",
    "torso_angle", "hip_displacement_x", "hip_displacement_y",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/gmdcsa24_final_deployment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Final epochs:", FINAL_EPOCHS)
print("Fall multiplier:", FALL_MULTIPLIER)
print("Decision threshold:", DECISION_THRESHOLD)

## 1. Đọc dữ liệu

Trong Kaggle, chọn **Add Input → Notebook Output** và gắn version đã lưu output của notebook 01.

In [ ]:
def find_kaggle_file(filename):
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            candidates.extend(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {filename}. Hãy Add Input output của notebook 01."
        )
    return sorted(candidates, key=lambda p: (len(p.parts), str(p)))[0]

FEATURE_PATH = find_kaggle_file("gmdcsa24_features_32.npz")
POSE_PATH = find_kaggle_file("gmdcsa24_pose_32.npz")
MANIFEST_PATH = find_kaggle_file("gmdcsa24_pose_manifest.csv")
FOLDS_PATH = find_kaggle_file("gmdcsa24_cross_subject_folds.csv")

feature_data = np.load(FEATURE_PATH, allow_pickle=False)
pose_data = np.load(POSE_PATH, allow_pickle=False)
manifest = pd.read_csv(MANIFEST_PATH)
saved_folds = pd.read_csv(FOLDS_PATH)

print("Feature:", FEATURE_PATH)
print("Pose:", POSE_PATH)
print("Manifest:", MANIFEST_PATH)
print("Folds:", FOLDS_PATH)
print("Feature arrays:", feature_data.files)
print("Pose arrays:", pose_data.files)

## 2. Ghép tensor đầu vào 8 kênh

Thứ tự kênh phải được giữ nguyên khi export model và triển khai web.

In [ ]:
def first_available_array(name):
    if name in feature_data.files:
        return feature_data[name]
    if name in pose_data.files:
        return pose_data[name]
    raise KeyError(f"Không tìm thấy mảng {name}")

def normalize_labels(values):
    values = np.asarray(values)
    if values.dtype.kind in "iufb":
        labels = values.astype(np.int64)
    else:
        labels = np.array([
            1 if "FALL" in str(value).upper()
            else 0 if "ADL" in str(value).upper()
            else -1
            for value in values
        ], dtype=np.int64)
    if not np.all(np.isin(labels, [0, 1])):
        raise ValueError("Label phải là ADL/0 hoặc FALL/1")
    return labels

def normalize_subjects(values):
    result = []
    for value in np.asarray(values):
        match = re.search(r"(\d+)", str(value))
        if match is None:
            raise ValueError(f"Không đọc được subject: {value}")
        result.append(int(match.group(1)))
    return np.asarray(result, dtype=np.int64)

skeleton = feature_data["skeleton_features"].astype(np.float32)
motion = feature_data["motion_features"].astype(np.float32)
global_features = feature_data["global_features"].astype(np.float32)

y = normalize_labels(first_available_array("labels"))
subjects = normalize_subjects(first_available_array("subjects"))
video_ids = first_available_array("video_ids").astype(str)

# 5 kênh theo từng khớp.
joint_channels = np.stack([
    skeleton[..., 0],
    skeleton[..., 1],
    skeleton[..., 3],
    motion[..., 0],
    motion[..., 1],
], axis=-1)

# Theo cấu trúc global_features của notebook 01:
# 0:2 = vị trí tâm hông; 6 = góc thân.
hip_position = global_features[..., 0:2]
hip_displacement = hip_position - hip_position[:, :1, :]
torso_angle = global_features[..., 6:7]

body_channels = np.concatenate(
    [torso_angle, hip_displacement], axis=-1
)
body_channels = np.repeat(
    body_channels[:, :, None, :], skeleton.shape[2], axis=2
)

# (N,T,V,C) -> (N,C,T,V)
X = np.concatenate([joint_channels, body_channels], axis=-1)
X = np.transpose(X, (0, 3, 1, 2)).astype(np.float32)

assert X.shape == (160, 8, 32, 33), X.shape
assert len(y) == len(subjects) == len(video_ids) == len(X)
assert np.isfinite(X).all()

print("X:", X.shape)
print("ADL:", int((y == 0).sum()), "| FALL:", int((y == 1).sum()))
print("Subjects:", np.unique(subjects, return_counts=True))
print("Finite:", np.isfinite(X).all())
print("Channels:", CHANNEL_NAMES)

## 3. Graph MediaPipe và augmentation

Augmentation chỉ áp dụng cho training loader. Validation và test luôn dùng dữ liệu gốc.

In [ ]:
POSE_EDGES = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
    (27,29),(28,30),(29,31),(30,32),(27,31),(28,32),
]

LEFT_RIGHT_PAIRS = [
    (1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
    (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32),
]

def make_normalized_adjacency(num_nodes=33):
    adjacency = np.eye(num_nodes, dtype=np.float32)
    for source, target in POSE_EDGES:
        adjacency[source, target] = 1.0
        adjacency[target, source] = 1.0
    degree = adjacency.sum(axis=1)
    inv_sqrt = np.power(np.maximum(degree, 1e-8), -0.5)
    adjacency = inv_sqrt[:, None] * adjacency * inv_sqrt[None, :]
    return torch.tensor(adjacency, dtype=torch.float32)

ADJACENCY = make_normalized_adjacency()

def augment_sequence(sequence):
    sequence = sequence.clone()
    _, num_frames, _ = sequence.shape

    # Crop 80–100% thời gian rồi nội suy lại 32 frame.
    if random.random() < 0.50:
        crop_length = random.randint(max(8, int(0.80 * num_frames)), num_frames)
        start = random.randint(0, num_frames - crop_length)
        crop = sequence[:, start:start + crop_length, :]
        sequence = F.interpolate(
            crop.permute(2, 0, 1),
            size=num_frames,
            mode="linear",
            align_corners=False,
        ).permute(1, 2, 0)

    # Lật ngang và hoán đổi khớp trái-phải.
    if random.random() < 0.50:
        original = sequence.clone()
        for left, right in LEFT_RIGHT_PAIRS:
            sequence[:, :, left] = original[:, :, right]
            sequence[:, :, right] = original[:, :, left]
        sequence[[0, 3, 5, 6]] *= -1.0

    # Nhiễu nhẹ trên tọa độ x, y.
    if random.random() < 0.35:
        sequence[0:2] += torch.randn_like(sequence[0:2]) * 0.01

    # Bỏ ngẫu nhiên 1–3 joint.
    if random.random() < 0.20:
        joint_ids = torch.randperm(sequence.shape[2])[:random.randint(1, 3)]
        sequence[:, :, joint_ids] = 0.0

    return sequence

class PoseSequenceDataset(Dataset):
    def __init__(self, features, labels, augment=False):
        self.features = torch.from_numpy(features).float()
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        sequence = self.features[index]
        if self.augment:
            sequence = augment_sequence(sequence)
        return sequence, self.labels[index]

def make_loader(indices, augment, shuffle, seed):
    dataset = PoseSequenceDataset(X[indices], y[indices], augment=augment)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )

print("Adjacency:", tuple(ADJACENCY.shape))
print("Số cạnh:", len(POSE_EDGES))

## 4. Mô hình ME-STGCN-TA

Kiến trúc một luồng: `8→64→64→128→128`, spatial average và temporal attention pooling.

In [ ]:
class SpatialGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.register_buffer("adjacency", adjacency.clone())
        self.edge_importance = nn.Parameter(torch.ones_like(adjacency))

    def forward(self, x):
        x = self.projection(x)
        graph = self.adjacency * self.edge_importance
        return torch.einsum("nctv,vw->nctw", x, graph)

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency, stride=1, dropout=0.0):
        super().__init__()
        self.graph_conv = SpatialGraphConv(in_channels, out_channels, adjacency)
        self.temporal_conv = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0),
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )
        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.temporal_conv(self.graph_conv(x)) + self.residual(x))

class MESTGCNTA(nn.Module):
    def __init__(self, adjacency, in_channels=8, num_classes=2, dropout=DROPOUT):
        super().__init__()
        num_nodes = adjacency.shape[0]
        self.data_bn = nn.BatchNorm1d(in_channels * num_nodes)
        self.blocks = nn.Sequential(
            STGCNBlock(in_channels, 64, adjacency, stride=1, dropout=dropout),
            STGCNBlock(64, 64, adjacency, stride=1, dropout=dropout),
            STGCNBlock(64, 128, adjacency, stride=2, dropout=dropout),
            STGCNBlock(128, 128, adjacency, stride=1, dropout=dropout),
        )
        self.temporal_attention = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(64, 1, kernel_size=1),
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        batch_size, channels, frames, nodes = x.shape
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, nodes * channels, frames)
        x = self.data_bn(x)
        x = x.view(batch_size, nodes, channels, frames).permute(0, 2, 3, 1).contiguous()
        x = self.blocks(x)
        temporal_features = x.mean(dim=3)
        attention = torch.softmax(self.temporal_attention(temporal_features), dim=2)
        video_features = (temporal_features * attention).sum(dim=2)
        logits = self.classifier(video_features)
        return logits, attention.squeeze(1)

model_check = MESTGCNTA(ADJACENCY).to(DEVICE)
with torch.no_grad():
    logits_check, attention_check = model_check(torch.zeros(2, 8, 32, 33, device=DEVICE))
print("Logits:", tuple(logits_check.shape))
print("Attention:", tuple(attention_check.shape))
print("Số tham số:", f"{sum(p.numel() for p in model_check.parameters()):,}")
del model_check
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Huấn luyện final model trên toàn bộ 160 video

**Vì sao không dùng 30–50 epoch?** Trong 04b, các epoch được inner validation chọn là 17, 3, 13 và 11. Trung vị là 12. Với 160 mẫu và batch 16, 12 epoch tương đương khoảng 120 optimizer updates; 30–50 epoch sẽ tạo 300–500 updates mà không có validation để phát hiện overfitting. Nhiều epoch hơn không đồng nghĩa model tốt hơn.

In [ ]:
all_indices = np.arange(len(y))
class_counts = np.bincount(y, minlength=2).astype(np.float32)
class_weights = len(y) / (2.0 * class_counts)
class_weights[1] *= FALL_MULTIPLIER
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

train_loader = make_loader(
    all_indices,
    augment=True,
    shuffle=True,
    seed=SEED,
)

final_model = MESTGCNTA(ADJACENCY).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = []
start_time = time.time()

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()
    total_loss = 0.0
    total_items = 0

    for features, labels in train_loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = final_model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_items += len(labels)

    epoch_loss = total_loss / total_items
    history.append({"epoch": epoch, "train_loss": epoch_loss})
    print(f"Epoch {epoch:02d}/{FINAL_EPOCHS} | train_loss={epoch_loss:.6f}")

elapsed_seconds = time.time() - start_time
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

print("\nHoàn thành trong:", round(elapsed_seconds, 2), "giây")
print("Class counts [ADL, FALL]:", class_counts.astype(int).tolist())
print("Class weights:", class_weights.tolist())

## 6. Kiểm tra kỹ thuật và lưu checkpoint PyTorch

Phần kiểm tra dưới đây chỉ xác nhận model chạy được và đầu ra hữu hạn. Không xem metric trên training set là bằng chứng tổng quát hóa.

In [ ]:
final_model.eval()
with torch.no_grad():
    sample_tensor = torch.from_numpy(X[:4]).float().to(DEVICE)
    sample_logits, sample_attention = final_model(sample_tensor)
    sample_probabilities = torch.softmax(sample_logits, dim=1)[:, 1]

assert torch.isfinite(sample_logits).all()
assert torch.isfinite(sample_attention).all()
assert torch.isfinite(sample_probabilities).all()

checkpoint_path = OUTPUT_DIR / "me_stgcn_ta_final_all_subjects.pt"
checkpoint = {
    "model_state_dict": {
        name: value.detach().cpu()
        for name, value in final_model.state_dict().items()
    },
    "architecture": "ME-STGCN-TA",
    "adjacency": ADJACENCY.cpu(),
    "input_shape": [None, 8, 32, 33],
    "channel_names": CHANNEL_NAMES,
    "sequence_length": 32,
    "num_joints": 33,
    "label_mapping": {"ADL": 0, "FALL": 1},
    "fall_multiplier": FALL_MULTIPLIER,
    "decision_threshold": DECISION_THRESHOLD,
    "epochs": FINAL_EPOCHS,
    "seed": SEED,
    "training_samples": int(len(y)),
    "training_subjects": sorted(np.unique(subjects).tolist()),
    "source_evaluation": {
        "protocol": "04b Nested LOSO",
        "accuracy": 0.8812,
        "sensitivity": 0.9747,
        "specificity": 0.7901,
        "false_negatives": 2,
        "false_positives": 17,
    },
}
torch.save(checkpoint, checkpoint_path)
np.save(OUTPUT_DIR / "mediapipe_adjacency_33.npy", ADJACENCY.numpy())
np.save(OUTPUT_DIR / "sample_input.npy", X[:1])

print("Sample probabilities:", sample_probabilities.cpu().numpy())
print("Attention shape:", tuple(sample_attention.shape))
print("Đã lưu:", checkpoint_path)

## 7. Export ONNX cho web

ONNX chỉ nhận tensor `(batch, 8, 32, 33)` và trả trực tiếp xác suất Fall. Cell tự cài `onnx`/`onnxruntime` nếu Kaggle image chưa có.

In [ ]:
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package_name = pip_name or import_name
        print(f"Đang cài {package_name}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package_name
        ])

ensure_package("onnx")
ensure_package("onnxruntime")

import onnx
import onnxruntime as ort

class DeploymentModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pose_features):
        logits, _ = self.model(pose_features)
        return torch.softmax(logits, dim=1)[:, 1]

deployment_model = DeploymentModel(final_model).to(DEVICE).eval()
dummy_input = torch.zeros(1, 8, 32, 33, dtype=torch.float32, device=DEVICE)
onnx_path = OUTPUT_DIR / "me_stgcn_ta_final_all_subjects.onnx"

torch.onnx.export(
    deployment_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["pose_features"],
    output_names=["fall_probability"],
    dynamic_axes={
        "pose_features": {0: "batch_size"},
        "fall_probability": {0: "batch_size"},
    },
    dynamo=False,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print("ONNX hợp lệ:", onnx_path)
print("Dung lượng:", round(onnx_path.stat().st_size / 1024 / 1024, 2), "MB")

## 8. So sánh kết quả PyTorch và ONNX

Nếu sai số lớn hơn `1e-4`, không dùng file ONNX để triển khai.

In [ ]:
verification_input = X[:8].astype(np.float32)
with torch.no_grad():
    pytorch_output = deployment_model(
        torch.from_numpy(verification_input).to(DEVICE)
    ).cpu().numpy()

session = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"],
)
onnx_output = session.run(
    ["fall_probability"],
    {"pose_features": verification_input},
)[0]

max_absolute_error = float(np.max(np.abs(pytorch_output - onnx_output)))
print("PyTorch:", pytorch_output)
print("ONNX:", onnx_output)
print("Max absolute error:", max_absolute_error)

if max_absolute_error > 1e-4:
    raise RuntimeError("ONNX không khớp PyTorch; chưa được triển khai web")
print("PyTorch–ONNX parity: PASS")

## 9. Metadata bắt buộc cho web

Model không nhận raw video. Web phải chạy MediaPipe, tạo đúng 8 kênh theo đúng thứ tự, rồi mới gọi ONNX.

In [ ]:
deployment_config = {
    "model_name": "ME-STGCN-TA final all subjects",
    "model_file": "me_stgcn_ta_final_all_subjects.onnx",
    "pytorch_file": "me_stgcn_ta_final_all_subjects.pt",
    "input_name": "pose_features",
    "output_name": "fall_probability",
    "input_layout": "NCTV",
    "input_shape": [None, 8, 32, 33],
    "dtype": "float32",
    "channel_names": CHANNEL_NAMES,
    "sequence_length": 32,
    "num_joints": 33,
    "decision_threshold": DECISION_THRESHOLD,
    "label_mapping": {"0": "ADL", "1": "FALL"},
    "feature_definition": {
        "relative_xy": "hip-centered and divided by per-video torso scale",
        "visibility": "MediaPipe visibility",
        "joint_velocity_xy": "temporal difference of normalized joint coordinates",
        "torso_angle": "global_features index 6, repeated over 33 joints",
        "hip_displacement_xy": "hip position minus first-frame hip position, repeated over 33 joints",
    },
    "training": {
        "samples": int(len(y)),
        "subjects": sorted(np.unique(subjects).tolist()),
        "epochs": FINAL_EPOCHS,
        "fall_multiplier": FALL_MULTIPLIER,
        "seed": SEED,
    },
    "evaluation_reference": {
        "notebook": "04b-gmdcsa24-me-stgcn-ta-sensitivity",
        "protocol": "Nested LOSO",
        "accuracy": 0.8812,
        "sensitivity": 0.9747,
        "specificity": 0.7901,
        "false_negatives": 2,
        "false_positives": 17,
    },
    "realtime_warning": (
        "Training clips were resampled as complete clips to 32 frames. "
        "A fixed realtime window length must be validated before claiming realtime performance."
    ),
}

config_path = OUTPUT_DIR / "model_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(deployment_config, file, indent=2, ensure_ascii=False)

readme_text = f"""GMDCSA24 ME-STGCN-TA deployment bundle

Files
- me_stgcn_ta_final_all_subjects.pt: PyTorch checkpoint
- me_stgcn_ta_final_all_subjects.onnx: browser/backend inference model
- model_config.json: input channels, threshold and metadata
- mediapipe_adjacency_33.npy: graph adjacency
- sample_input.npy: one valid input tensor with shape (1,8,32,33)
- training_history.csv: final training loss

Important
- Decision threshold: {DECISION_THRESHOLD}
- The model input is not raw RGB video.
- MediaPipe preprocessing must exactly reproduce the 8 channels.
- Nested LOSO 04b is the evaluation evidence. This final model has no independent test set.
- Realtime temporal-window duration still requires validation.
"""
(OUTPUT_DIR / "README.txt").write_text(readme_text, encoding="utf-8")

bundle_path = shutil.make_archive(
    "/kaggle/working/gmdcsa24_me_stgcn_ta_final_deployment",
    "zip",
    OUTPUT_DIR,
)

print("Config:", config_path)
print("Bundle:", bundle_path)
print("Các file:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", file_path.name, f"({file_path.stat().st_size / 1024:.1f} KB)")

## Lưu version

Sau khi chạy thành công:

1. **Save Version → Quick Save**.
2. Version name: `v1-final-all-subjects-onnx`.
3. Bật **Always save output when creating a Quick Save**.
4. Tải `gmdcsa24_me_stgcn_ta_final_deployment.zip`.

Không báo cáo accuracy đo trên chính 160 training videos. Báo cáo chính vẫn lấy từ notebook 04b Nested LOSO.